<a href="https://colab.research.google.com/github/khalid-saqr/picoNewton/blob/main/waveform_susceptibility/notebooks/waveform_susceptibility_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Harmonic interactions shape anisotropy-induced transverse force in arterial blood flow

This notebook is the clean, executable Google Colab workflow for `piconewton-waveform-susceptibility` version 1.0.1. It mounts Google Drive, records the exact repository commit, installs the verified parent solver and manuscript package, runs the publication-resolution analysis, verifies every declared output and headline statistic, writes SHA-256 checksums, creates a portable archive, and displays manuscript Figures 1--5 and Supplementary Figure S1.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from uuid import uuid4

REPOSITORY_URL = 'https://github.com/khalid-saqr/picoNewton.git'
REPOSITORY_REF = os.environ.get('PICONEWTON_REF', 'main')
SESSION_ID = (
    datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    + '_'
    + uuid4().hex[:8]
)
LOCAL_ROOT = Path('/content') / f'picoNewton_{SESSION_ID}'
RUN_ROOT = (
    Path('/content/drive/MyDrive/picoNewton_waveform_susceptibility/runs')
    / SESSION_ID
)
ANALYSIS_ROOT = RUN_ROOT / 'analysis'
RUN_ROOT.mkdir(parents=True, exist_ok=False)

subprocess.run(
    ['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(LOCAL_ROOT)],
    check=True,
)
subprocess.run(
    ['git', 'checkout', REPOSITORY_REF], cwd=LOCAL_ROOT, check=True
)
COMMIT_SHA = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=LOCAL_ROOT, text=True
).strip()
print('Run directory:', RUN_ROOT)
print('Repository commit:', COMMIT_SHA)

In [ ]:
def run(command, cwd=None):
    print('+', ' '.join(str(item) for item in command))
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)

run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
run([
    sys.executable, '-m', 'pip', 'install', '-e',
    str(LOCAL_ROOT / 'picoNewton_v3'),
])
run([
    sys.executable, '-m', 'pip', 'install', '-e',
    str(LOCAL_ROOT / 'waveform_susceptibility'),
])

In [ ]:
metadata = {
    'session_id': SESSION_ID,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'repository_url': REPOSITORY_URL,
    'repository_ref': REPOSITORY_REF,
    'commit_sha': COMMIT_SHA,
    'python': sys.version,
    'platform': platform.platform(),
    'analysis_root': str(ANALYSIS_ROOT),
}
(RUN_ROOT / 'runtime_metadata.json').write_text(
    json.dumps(metadata, indent=2, sort_keys=True),
    encoding='utf-8',
)
metadata

In [ ]:
run([
    'piconewton-waveform-susceptibility',
    '--output', str(ANALYSIS_ROOT),
    '--radial-order', '150',
    '--time-points', '2048',
    '--quadrature-nodes', '256',
    '--validation-epsilon', '0.08',
    '--figure-dpi', '600',
])

In [ ]:
from IPython.display import Image, display

table_names = [
    'artery_atlas.csv',
    'crossed_susceptibility.csv',
    'waveform_controls.csv',
    'harmonic_pair_attribution.csv',
    'reduced_law_validation.csv',
    'constitutive_robustness.csv',
]
figure_names = ['Figure1', 'Figure2', 'Figure3', 'Figure4', 'Figure5', 'FigureS1']
required = [
    ANALYSIS_ROOT / 'analysis_summary.json',
    ANALYSIS_ROOT / 'operator_archive.npz',
    ANALYSIS_ROOT / 'figures' / 'figure_manifest.json',
    *[ANALYSIS_ROOT / name for name in table_names],
    *[
        ANALYSIS_ROOT / 'figures' / f'{name}.{suffix}'
        for name in figure_names
        for suffix in ('pdf', 'svg', 'png')
    ],
]
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise RuntimeError(f'Missing or empty outputs: {missing}')

summary = json.loads(
    (ANALYSIS_ROOT / 'analysis_summary.json').read_text(encoding='utf-8')
)
assert summary['configuration'] == {
    'radial_order': 150,
    'time_points': 2048,
    'quadrature_nodes': 256,
    'validation_epsilon': 0.08,
    'harmonics': 6,
}
assert summary['arteries'] == 6
assert summary['crossed_entries'] == 36
assert summary['operator_samples'] == 12
assert summary['held_out_predictions'] == 1068
assert summary['constitutive_paths'] == 9
law = summary['reduced_law']
assert abs(law['retained_energy'] - 0.9999860359) < 1e-9
assert abs(law['median_relative_error'] - 0.02231464) < 1e-7
assert abs(law['p90_relative_error'] - 0.10188632) < 1e-7
assert abs(law['maximum_relative_error'] - 0.16296206) < 1e-7

checksum_targets = [RUN_ROOT / 'runtime_metadata.json', *sorted(ANALYSIS_ROOT.rglob('*'))]
checksum_lines = []
for path in checksum_targets:
    if path.is_file():
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        checksum_lines.append(f'{digest}  {path.relative_to(RUN_ROOT)}')
(RUN_ROOT / 'checksums.sha256').write_text(
    '\n'.join(checksum_lines) + '\n', encoding='utf-8'
)
archive_path = RUN_ROOT / 'waveform_susceptibility_results.zip'
temporary_archive = shutil.make_archive(
    str(Path('/content') / f'waveform_susceptibility_results_{SESSION_ID}'),
    'zip',
    root_dir=RUN_ROOT,
)
shutil.move(temporary_archive, archive_path)
if not archive_path.is_file() or archive_path.stat().st_size == 0:
    raise RuntimeError('Result archive was not created.')

print(f'Verified {len(required)} required outputs.')
print('Checksums:', RUN_ROOT / 'checksums.sha256')
print('Archive:', archive_path)
for name in figure_names:
    display(Image(filename=str(ANALYSIS_ROOT / 'figures' / f'{name}.png')))